# 6. Using the data in MESA

MESA's colors module reads, from the `&colors` namelist:
- `stellar_atm`: a folder holding a 3-axis `flux_cube.bin` and `lookup_table.csv`;
- `instrument`: a filter-set folder holding its curves and the index file named after the folder;
- `vega_sed`: a Vega spectrum, for Vega magnitudes.

SED_Tools prints the right absolute paths after every download, build, combine and `mesa_prepare`. This notebook shows where they come from, and the one rule for split grids: **MESA reads one variant folder, never the split grid's own folder**. Point `stellar_atm` at the variant folder where it is, or export the variant with `mesa_prepare`; do not copy the grid's top folder into MESA.

In [ ]:
import numpy as np

from sed_tools import mesa_inlist_lines
from sed_tools.api import SED, Filters
from sed_tools.cube.fileio import read_header
from sed_tools.cube.mesa_export import export_variant, list_variants

KURUCZ_CUT = dict(teff_min=4000, teff_max=8000, logg_min=3.0, logg_max=5.0,
                  metallicity_min=-1.0, metallicity_max=0.5)
# Notebook 1 downloads these grids; they are fetched here only if missing, so this notebook needs no network.
for name, cut in (("Kurucz2003all", KURUCZ_CUT), ("bbody", {}),):
    if not (SED._model_root / name / "lookup_table.csv").is_file():
        SED.fetch(name, **cut)
twomass = Filters.fetch("2MASS", "2MASS")
stellar_models = SED._model_root
kurucz_dir = stellar_models / "Kurucz2003all"

## 1. The inlist lines

`mesa_inlist_lines(folder)` gives the `stellar_atm` or `instrument` line(s) for a local grid or filter set; `sed-tools mesa-path FOLDER` prints them. A grid that is not split gives one line; a split grid gives one per variant.

In [ ]:
for folder in (stellar_models / "bbody", kurucz_dir, twomass):
    print(*mesa_inlist_lines(folder), sep="\n")

In [ ]:
!sed-tools mesa-path {kurucz_dir}

## 2. The two cube layouts

- A variant's `flux_cube.bin` is the 3-axis cube MESA reads: four integers (n_Teff, n_logg, n_[M/H], n_λ), the axes, then the flux.
- The split grid's `flux_cube_nd.bin` is the all-axes cube SED_Tools reads: it starts with −1 and `FLUXCUBE`, names its axes, and has the extra axes (`alpha`) as further dimensions. MESA cannot read it, and the grid's folder deliberately has no `flux_cube.bin`, so MESA pointed there finds no cube.

In [ ]:
variant_dir = kurucz_dir / "Kurucz2003all_alpha_0p00_lh_1p25_vtur_2_km_s"
three_axis = read_header(variant_dir / "flux_cube.bin")
print("variant:", three_axis.names, [len(a) for a in three_axis.axes], len(three_axis.wavelengths), "wavelengths")
print("first bytes:", np.fromfile(variant_dir / "flux_cube.bin", dtype="<i4", count=4))

all_axes = read_header(kurucz_dir / "flux_cube_nd.bin")
print("all-axes:", all_axes.names, [len(a) for a in all_axes.axes], len(all_axes.wavelengths), "wavelengths")
print("first bytes:", (kurucz_dir / "flux_cube_nd.bin").read_bytes()[:12])

assert not three_axis.all_axes and all_axes.all_axes
assert not (kurucz_dir / "flux_cube.bin").exists()

## 3. Exporting one variant: `mesa_prepare`

`export_variant` (the command `sed-tools mesa_prepare`) writes one variant as a clean, self-contained folder: its spectra, its `lookup_table.csv` and its `flux_cube.bin`. Use it to move a variant elsewhere, for example into a MESA work directory.

In [ ]:
variants = list_variants(kurucz_dir)
for v in variants:
    print(v.label, v.extra_axes, v.n_spectra, "spectra,", v.flux_cube_path.parent.name)

In [ ]:
exported = export_variant(variants[1], SED._model_root.parent / "mesa_ready" / "Kurucz2003all_alpha04", overwrite=True)
print(sorted(p.name for p in exported.iterdir() if not p.name.endswith(".txt")))
print(mesa_inlist_lines(exported))

In [ ]:
!sed-tools mesa_prepare --grid Kurucz2003all --variant Kurucz2003all_alpha_0p00_lh_1p25_vtur_2_km_s --output {SED._model_root.parent / "mesa_ready" / "Kurucz2003all_alpha00"}

## 4. A `&colors` namelist

The lines above, with Vega from `Filters.fetch_vega()` (the command `sed-tools vega`), give the namelist. MESA reads relative paths against `MESA_DIR`, so the paths are absolute.

In [ ]:
vega = Filters.fetch_vega()
colors = "\n".join([
    "&colors",
    "    use_colors = .true.",
    "    " + mesa_inlist_lines(twomass)[0],
    "    " + mesa_inlist_lines(variant_dir)[0],
    f"    vega_sed = '{vega}'",
    "/",
])
print(colors)

## Command-line equivalents

| Python | Command line |
|---|---|
| `mesa_inlist_lines(folder)` | `sed-tools mesa-path FOLDER` |
| `export_variant(list_variants(grid)[i], out)` | `sed-tools mesa_prepare --grid GRID --variant VARIANT --output OUT` |
| `Filters.fetch_vega()` | `sed-tools vega` |